<a href="https://colab.research.google.com/github/maramgueye/fairness/blob/main/MiProjetGUEYE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mi-Projet: Analyse et Mitigation des Biais**
Par Maram Sall GUEYE

Ce notebook présente une analyse approfondie des métadonnées du dataset qui m'ont ete attribuee, avec identification des biais et application de méthodes de pré-processing pour les réduire.

## 1. Introduction


1.1 Contexte du cas d'usage

Le dataset Chest X ray NIH 14 est un ensemble de radiographies pulmonaires fourni par le NIH. Il contient des métadonnées riches sur les patients : âge, genre, nombre de passages radiologiques, et labels de 14 pathologies possibles (ou absence de maladie).

Dans ce mi-projet, nous travaillons uniquement sur les métadonnées d'un sous-ensemble de ~15 000 patients. L'objectif final (projet complet) sera d'entraîner un modèle de classification sur les images pour prédire si un patient est malade ou non.

1.2 Problématique éthique et biais potentiels

Comme vu en cours lors de la session 1, les biais en Machine Learning sont multi-factoriels et peuvent provenir de :
*   Le protocole de collecte : biais d'agrégation (paradoxe de Simpson), biais d'échantillonnage
*   Les données elles-mêmes : biais historiques, biais de représentation (certains groupes sous-représentés)
*   Les algorithmes : amplification des biais présents dans les données

Dans notre cas, les biais potentiels identifiés pourraient être les biais de genre et ceux liés à l'âge.

Ces biais peuvent mener à un modèle discriminant, moins précis pour les groupes sous-représentés, ce qui est particulièrement grave dans un contexte médical.

1.3 Objectif de l'analyse


*   Study the data, the distribution of each

*   feature and its relation to the target
Highlight some bias present in the data

*   Learn a basic machine learning model using logistic regression

*   Compute the confusion matrix and different fairness metrics





# Nouvelle section

# Debut du projet

1.1 Installation

In [ ]:
# To execute only in Colab
! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit

1.2 Import des Bibliothèques

In [ ]:
# Code to compute fairness metrics using aif360
#Fonction issus des TD

from aif360.sklearn.metrics import *
from sklearn.metrics import  balanced_accuracy_score


def get_metrics(
    y_true, # list or np.array of truth values
    y_pred=None,  # list or np.array of predictions
    prot_attr=None, # list or np.array of protected/sensitive attribute values
    priv_group=1, # value taken by the privileged group
    pos_label=1, # value taken by the positive truth/prediction
    sample_weight=None # list or np.array of weights value,
):
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["base_rate_preds"] = base_rate(
        y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred))>1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] =None
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
        y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# 2 . Préparation des Données

## 2.1 Chargement des données avec mon Google Drive

In [ ]:
import numpy as np
import fairlearn
np.__version__, fairlearn.__version__

('2.0.2', '0.13.0')

In [ ]:
from google.colab import files
import pandas as pd


uploaded = files.upload()
df = pd.read_csv('Gueye_Maram_Sall.csv')



Saving Gueye_Maram_Sall.csv to Gueye_Maram_Sall (1).csv


In [ ]:
print("Données chargées:", len(df),"échantillons")
print("Colonnes:", list(df.columns))
print("\nAperçu des données:")
df.head(10)

Données chargées: 54280 échantillons
Colonnes: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]']

Aperçu des données:


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143
5,00000003_001.png,Hernia,1,3,74,F,PA,2500,2048,0.168,0.168
6,00000003_002.png,Hernia,2,3,75,F,PA,2048,2500,0.168,0.168
7,00000003_003.png,Hernia|Infiltration,3,3,76,F,PA,2698,2991,0.143,0.143
8,00000003_004.png,Hernia,4,3,77,F,PA,2500,2048,0.168,0.168
9,00000003_005.png,Hernia,5,3,78,F,PA,2686,2991,0.143,0.143


### Informations générales sur le dataset

In [ ]:
print("Nombre total d'échantillons:",len(df))
print("Nombre de colonnes:",len(df.columns))
print("\nTypes des données:")
print(df.dtypes)
print("\nValeurs manquantes:")
print(df.isnull().sum())
print("\nDescription:")
df.describe()

Nombre total d'échantillons: 54280
Nombre de colonnes: 11

Types des données:
Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
dtype: object

Valeurs manquantes:
Image Index                    0
Finding Labels                 0
Follow-up #                    0
Patient ID                     0
Patient Age                    0
Patient Gender                 0
View Position                  0
OriginalImage[Width            0
Height]                        0
OriginalImagePixelSpacing[x    0
y]                             0
dtype: int64

Description:


,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
count,54280.000000,54280.000000,54280.000000,54280.000000,54280.000000,54280.000000,54280.000000
mean,8.367668,14131.675571,47.009046,2649.411791,2488.705306,0.155449,0.155449
std,14.646503,8385.303176,17.130411,340.706652,400.521233,0.016058,0.016058
min,0.000000,1.000000,1.000000,1189.000000,966.000000,0.115000,0.115000
25%,0.000000,7034.000000,34.000000,2500.000000,2048.000000,0.143000,0.143000
50%,3.000000,13662.000000,49.000000,2530.000000,2544.000000,0.143000,0.143000
75%,9.000000,20505.000000,60.000000,2992.000000,2991.000000,0.168000,0.168000
max,157.000000,30803.000000,414.000000,3827.000000,3567.000000,0.198800,0.198800


## Nettoyage et Préparation des Données

### 1. Traitement des age aberrants

In [ ]:
print("\n1. Analyse de l'âge:")
print("   - Min:", df['Patient Age'].min())
print("   - Max:", df['Patient Age'].max())
print("   - Moyenne:", df['Patient Age'].mean())
print("   - Médiane:", df['Patient Age'].median())


1. Analyse de l'âge:
   - Min: 1
   - Max: 414
   - Moyenne: 47.0090456890199
   - Médiane: 49.0


En observant la distribution de Patient Age, on constate un maximum de 414 ans, biologiquement impossible car Ethel Caterham, la doyenne de l'humanité, vient de fêter ses 116 ans !

Nous allons donc filtrer les âges aberrants et fixer une limite d'âge à 120 ans.
L'âge minimal n'a pas besoin d'être changé, car nous voyons bien que 1 an est un âge classique.

In [ ]:
df_clean = df[(df['Patient Age'] <= 120)].copy()
print("Nombres de ligne supprimées car âge aberrant:",len(df) - len(df_clean))

Nombres de ligne supprimées car âge aberrant: 8


Après nettoyage de ces données, 8 lignes ont été supprimées, car les âges indiqués dépassaient 120 ans, ce qui, comme nous l'avons dit précédemment, est biologiquement impossible. Il s'agissait donc d'une erreur.

2. Traitement des valeurs manquantes

In [ ]:
missing_avant = df_clean.isnull().sum().sum()
df_clean = df_clean.dropna()
missing_apres = df_clean.isnull().sum().sum()
print(f"Valeurs manquantes supprimées: {missing_avant - missing_apres}")

Valeurs manquantes supprimées: 0


### 3. Doublons liés au Patient ID

### un patient = plusieurs radiographies

Dans le jeu de données NIH Chest X-Ray, chaque ligne correspond à une radiographie et non à un patient, ce qui n'est pas idéal pour une analyse. Un même patient peut avoir effectué plusieurs visites au cours du temps (ce que nous pouvons constater dans la colonne "Follow-up#" ), et donc apparaître plusieurs fois dans le jeu de données avec le même identifiant de patient.

### Problème

 - si on compte les radiographies et non les patients, les individus avec beaucoup de visites (souvent les plus malades et les plus âgés) sont sur-représentés. Cela fausse toute notre analyse descriptive et nos métriques de fairness.

### Visualisation du problème

In [ ]:
n_lignes   = len(df_clean)
n_patients = df_clean['Patient ID'].nunique()
print(f'Nombre de lignes dans notre dataset/nbre de radiographies : {n_lignes}')
print(f'Nombre de patients uniques       : {n_patients}')
print(f'Patients avec present plusieurs fois dans notre dataset   : {(df_clean["Patient ID"].value_counts() > 1).sum()}')



Nombre de lignes dans notre dataset/nbre de radiographies : 54272
Nombre de patients uniques       : 14998
Patients avec present plusieurs fois dans notre dataset   : 6414


### Nombre de visites par patient


In [ ]:
visits_per_patient = df_clean['Patient ID'].value_counts()
print(f'\nDistribution du nombre de visites par patient :')
print(visits_per_patient.value_counts().sort_index().head(10))
print(f'Maximum de visites pour un patient : {visits_per_patient.max()}')


Distribution du nombre de visites par patient :
count
1     8584
2     1954
3     1020
4      622
5      454
6      397
7      298
8      249
9      199
10     160
Name: count, dtype: int64
Maximum de visites pour un patient : 158


### Je décide de garder une ligne par patient.

Nous observons donc qu’un biais peut effectivement apparaître : si certains patients effectuent des dizaines de visites, ce sont très probablement les patients les plus malades qui se retrouvent surreprésentés dans nos données, alors qu’il ne s’agit que d’une seule et même personne.

Nous aurions pu conserver ces multiples visites pour analyser et atténuer ce biais. Cependant, si nous gardons plusieurs lignes pour un même patient, et que ce nombre de visites est réparti de façon aléatoire, nos analyses ultérieures risquent d’être faussées.

Prenons un exemple : si nous avons une population de deux personnes — un homme (trois visites) et une femme (une visite) —, en analysant les données ligne par ligne, nous pourrions conclure à tort qu’il y a quatre individus, avec trois quarts d’hommes et un quart de femmes.

De plus, la question “ce patient est-il malade ?” doit pouvoir recevoir une réponse unique par patient dans le jeu de données — sinon, l’étiquette devient ambiguë.

Nous aurions donc pu conserver l’ensemble des visites pour une analyse longitudinale des pathologies (biais temporel). Toutefois, pour notre objectif de classification binaire (malade / sain) et pour garantir l'integrite des métriques de fairness, nous avons fait le choix de dédoublonner les données. Nous retenons la visite de suivi la plus récente (follow‑up # maximum), car elle correspond au diagnostic le plus à jour pour chaque patient.

In [ ]:
if 'Follow-up #' in df_clean.columns:
    df_clean = (df_clean
                .sort_values(['Patient ID', 'Follow-up #'], ascending=[True, False])
                .drop_duplicates(subset='Patient ID', keep='first')
                .reset_index(drop=True))
    print('Stratégie : dernière visite conservée (Follow-up # max)')
else:
    df_clean = (df_clean
                .drop_duplicates(subset='Patient ID', keep='first')
                .reset_index(drop=True))
    print('Stratégie : première occurrence conservée')

assert df_clean['Patient ID'].duplicated().sum() == 0
print(f'\nAvant déduplication : {n_avant} lignes')
print(f'Apres déduplication : {len(df_clean)} lignes')
print(f'Supprimées          : {n_avant - len(df_clean)} ({(n_avant-len(df_clean))/n_avant*100:.1f}%)')
print(f'\ndf_clean = {len(df_clean)} patients uniques')

Stratégie : dernière visite conservée (Follow-up # max)

Avant déduplication : 54272 lignes
Apres déduplication : 14998 lignes
Supprimées          : 39274 (72.4%)

df_clean = 14998 patients uniques


### 4. View Position

Le jeu de données contient deux positions de vue : PA (patient debout) et AP (patient allongé, souvent dans des cas graves). Si la position AP est sur-représentée chez les malades, il s'agit d'un biais de mesure. Cette catégorie sera étudiée plus en détail dans la partie consacrée à l'observation des biais.

### 5. Transformation de nos données en données numériques pour en simplifier l'analyse.

Dans les précédents TP, notamment le TP1 et le TP2, nous avons utilisé l'encodage one-hot, notamment pour analyser le genre. Or, nous constatons que, pour nos données, le label « Finding Labels » peut prendre plusieurs formes différentes.
Je considère que, dès lors que nous obtenons une réponse dans la rubrique « Finding Labels », c'est-à-dire que nous avons trouvé quelque chose sur la radiographie du thorax, alors nous pouvons dire que c'est bon, quel que soit le résultat.
Notre variable « Has_disease » sera alors binaire.
Par exemple, si un patient a trois maladies et un autre « seulement » une, ils seront traités de la même façon.

In [ ]:
df_clean['Has_Disease'] = (df_clean['Finding Labels'] != 'No Finding').astype(int)
df_clean['Gender_Binary'] = (df_clean['Patient Gender'] == 'M').astype(int)
print("'Has_Disease': 0=sain et 1=malade")
print("'Gender_Binary': 0=F et 1=M")

'Has_Disease': 0=sain et 1=malade
'Gender_Binary': 0=F et 1=M


### Visualisation du jeu de données nettoyé.

In [ ]:
df_clean.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Has_Disease,Gender_Binary
0,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,1,1
1,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,0,1
2,00000003_007.png,Hernia,7,3,80,F,PA,2582,2905,0.143,0.143,1,0
3,00000004_000.png,Mass|Nodule,0,4,82,M,AP,2500,2048,0.168,0.168,1,1
4,00000005_007.png,Effusion|Infiltration,7,5,70,F,PA,2566,2681,0.143,0.143,1,0


# Analyse descriptive et observation des bias

## 1. Analyse univariée

Nous allons commencer par observer la distribution de nos différentes valeurs démographiques et en faire des répartitions afin de voir s'il n'existe pas de disparité trop importante.

1. Distribution du genre

In [ ]:
gender_dist = df_clean['Patient Gender'].value_counts()
print(gender_dist)

Patient Gender
M    8142
F    6856
Name: count, dtype: int64


In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Distribution par Genre', 'Pourcentages'),
                    specs=[[{'type':'bar'}, {'type':'pie'}]])

fig.add_trace(go.Bar(x=gender_dist.index, y=gender_dist.values,
                     marker_color=['lightblue', 'lightpink']),
              row=1, col=1)

fig.add_trace(go.Pie(labels=gender_dist.index, values=gender_dist.values,
                     marker_colors=['lightblue', 'lightpink']),
              row=1, col=2)

fig.show()

Le dataset présente une légère sur-représentation masculine (environ 54,3% hommes et 45,7% femmes). Cela correspond à un biais de représentation. En effet, le groupe des femmes est sous-représenté par rapport à la population générale des patients susceptibles d'être radiographiés. Un modèle entraîné sur ces données risque d'être plus performant pour les hommes.

2. Distribution de l'age


In [ ]:
age_stats = df_clean['Patient Age'].describe()
print(age_stats)

count    14998.000000
mean        46.395653
std         16.814304
min          1.000000
25%         34.000000
50%         48.000000
75%         59.000000
max         94.000000
Name: Patient Age, dtype: float64


In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Histogramme de l\'age', 'Boxplot de l\'age'),
                    specs=[[{'type': 'histogram'}, {'type': 'box'}]])
fig.add_trace(go.Histogram(x=df_clean['Patient Age'], nbinsx=40,
                            marker_color='steelblue', name='Age'), row=1, col=1)
fig.add_trace(go.Box(y=df_clean['Patient Age'], marker_color='steelblue',
                     name='Age', boxmean=True), row=1, col=2)
fig.update_layout(title='Distribution de l\'Age', height=400, showlegend=False)
fig.show()



La distribution des âges est centrée autour de 47 ans et le maximum est de 47, ce qui pourrait faire croire a uene bonne repartition. Toutefois, les tranches 0-10 ans et 80+ sont nettement sous-représentées par rapport aux tranches 40-70 ans. C'est un biais d'échantillonnage. En effet, le modèle futur risque d'être moins robuste pour les enfants et les très âgés. La moyenne et la médiane sont proches, ce qui indique une distribution approximativement symétrique.

3. Distribution des maladies

In [ ]:
disease_dist = df_clean['Finding Labels'].value_counts()
disease_pct = df_clean['Finding Labels'].value_counts(normalize=True) * 100
print(f"Top 10 diagnostiques les plus fréquents:")
for i, (disease, count) in enumerate(disease_dist.head(10).items(), 1):
    print(f"  {i:2d}. {disease:20s}: {count:5d} ({disease_pct[disease]:5.2f}%)")

Top 10 diagnostiques les plus fréquents:
   1. No Finding          :  9919 (66.14%)
   2. Infiltration        :  1236 ( 8.24%)
   3. Atelectasis         :   478 ( 3.19%)
   4. Nodule              :   418 ( 2.79%)
   5. Effusion            :   303 ( 2.02%)
   6. Mass                :   224 ( 1.49%)
   7. Cardiomegaly        :   197 ( 1.31%)
   8. Pleural_Thickening  :   175 ( 1.17%)
   9. Fibrosis            :   128 ( 0.85%)
  10. Atelectasis|Effusion:   110 ( 0.73%)


Nous constatons que No finding est le diagnostique le plus frequent,c e qui est coherent car il s'agit de despistage?

In [ ]:
disease_binary = df_clean['Has_Disease'].value_counts()
print(disease_binary)

Has_Disease
0    9919
1    5079
Name: count, dtype: int64


Pour la distribution entre les malades et les sains, le ratio est de 54/46 en faveur des sains, ce qui est négligeable et ne déséquilibre pas les données.

Visualisation

In [ ]:
fig = px.bar(x=disease_dist.head(10).index,
             y=disease_dist.head(10).values,
             labels={'x': 'Diagnostique', 'y': 'Nombre de cas'},
             title='Top 10 des Diagnostics les plus fréquents',
             color=disease_dist.head(10).values,
             color_continuous_scale='Viridis')
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

Analyse de la position

In [ ]:
df_clean['View_Binary'] = (df_clean['View Position'] == 'AP').astype(int)

print(f"Position PA (0) : {(df_clean['View_Binary'] == 0).sum()} patients")
print(f"Position AP (1) : {(df_clean['View_Binary'] == 1).sum()} patients")

Position PA (0) : 12387 patients
Position AP (1) : 2611 patients


Avec 12387 patients en position PA soit 82,6% de l'échantillon et seulement 2611 patients en position AP soit 17,4%, nous observons un déséquilibre important dans la répartition.  Ce déséquilibre de plus de 65 points de pourcentage entre les deux groupes est considérable et constitue en soi un biais de représentation important.

# 2. Analyse bivariée

l'analyse bivariée permet d'étudier les relations entre variables. Ici, nous croisons nos variables démographiques avec la variable cible Has_Disease pour identifier les biais de données.

## Analyse bivariée sur le genre
###(Genre et Maladie)


In [ ]:
cross_gender_disease = pd.crosstab(
    df_clean['Patient Gender'],
    df_clean['Has_Disease'],
    normalize='index'
) * 100

print("Taux de maladie par genre (en %):")
print(cross_gender_disease.round(2))

diff_disease_rate = abs(cross_gender_disease.loc['M', 1] - cross_gender_disease.loc['F', 1])
print(f"\n📊 Différence de taux de maladie entre genres: {diff_disease_rate:.2f}%")



Taux de maladie par genre (en %):
Has_Disease         0      1
Patient Gender              
F               66.20  33.80
M               66.08  33.92

📊 Différence de taux de maladie entre genres: 0.13%


Visualisation

In [ ]:
cross_counts = pd.crosstab(
    df_clean['Patient Gender'],
    df_clean['Has_Disease']
)

fig = go.Figure(data=[
    go.Bar(name='Sain', x=cross_counts.index, y=cross_counts[0]),
    go.Bar(name='Malade', x=cross_counts.index, y=cross_counts[1])
])

fig.update_layout(
    title='Distribution des Maladies par Genre',
    xaxis_title='Genre',
    yaxis_title='Nombre de patients',
    barmode='group',
    height=400
)
fig.show()

On observe une différence de taux de maladie entre les genres. Si cette différence peut partiellement s'expliquer médicalement, elle constitue un biais de genre dans les données. Ce biais, s'il n'est pas corrigé, sera capté par le modèle et amplifié> Il peut s'agir d'un biais historique.

#Analyse bivariée sur l'age
###(Age et Maladie)


Nous suivrons la methode vu en cours avec les Boxplot, ici nous alors observer lâge selon statut

In [ ]:
fig = px.box(df_clean, x='Has_Disease', y='Patient Age',
             color='Has_Disease',
             labels={'Has_Disease': 'Statut', 'Patient Age': 'Age'},
             title='Distribution de lAge selon le Statut (Sain/Malade)',
             color_discrete_map={0: 'mediumseagreen', 1: 'tomato'})
fig.update_xaxes(tickvals=[0, 1], ticktext=['Sain', 'Malade'])
fig.update_layout(height=400, showlegend=False)
fig.show()


In [ ]:
corr_pearson = np.corrcoef(df_clean['Patient Age'].values,
                            df_clean['Has_Disease'].values)[0][1]
print(f'Corrélation de Pearson (Age et Maladie) : {corr_pearson:.4f}')

Corrélation de Pearson (Age et Maladie) : 0.1493


0.1493 est une valeur faible mais positive> Ainsi, il existe une très légère tendance : plus les patients sont âgés, plus ils ont tendance à être malades. Cependant, cette relation est très faible, donc'âge n'explique quasiment pas la variance du statut de maladie


Scatter plot âge vs probabilité de maladie (moyenne glissante)

In [ ]:
age_disease = (df_clean.groupby('Patient Age')['Has_Disease']
               .agg(['mean', 'count'])
               .reset_index())
age_disease.columns = ['Age', 'Taux_Maladie', 'Effectif']

fig = px.scatter(age_disease, x='Age', y='Taux_Maladie',
                 size='Effectif', size_max=20,
                 labels={'Age': 'Age', 'Taux_Maladie': 'Taux de maladie'},
                 title='Taux de maladie par age (taille = effectif à cet age)',
                 trendline='lowess',
                 color_discrete_sequence=['steelblue'])
fig.update_layout(height=450)
fig.show()

### Analyse croisée : Genre × Âge × Maladie

*Un paradoxe de Simpson est un phénomène statistique où une tendance observée dans la population globale peut s'inverser ou disparaître lorsqu'on analyse les sous-groupes.*
Pour vérifier l'absence de paradoxe de Simpson, nous analysons le taux de maladie simultanément selon le genre et la tranche d'âge.

Visualisons le taux de maladie par age et genre

In [ ]:
age_deciles = pd.qcut(df_clean['Patient Age'], q=10, duplicates='drop')

cross_age_gender = (df_clean
                    .groupby([age_deciles, 'Patient Gender'])['Has_Disease']
                    .mean().unstack() * 100)
print(cross_age_gender.round(1).to_string())



Patient Gender     F     M
Patient Age               
(0.999, 23.0]   25.2  27.0
(23.0, 31.0]    24.7  26.2
(31.0, 37.0]    27.4  22.8
(37.0, 43.0]    27.0  26.3
(43.0, 48.0]    31.9  31.3
(48.0, 52.0]    34.3  37.0
(52.0, 57.0]    40.9  38.0
(57.0, 61.0]    39.3  38.6
(61.0, 67.0]    46.6  43.8
(67.0, 94.0]    46.7  45.5


/tmp/ipython-input-3722099502.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
melted = cross_age_gender.reset_index().melt(
    id_vars='Patient Age', var_name='Genre', value_name='Taux_Maladie')
melted['Age_label'] = melted['Patient Age'].astype(str)

fig = px.bar(melted, x='Age_label', y='Taux_Maladie', color='Genre',
             barmode='group',
             labels={'Age_label': 'Décile dage', 'Taux_Maladie': 'Taux maladie (%)'},
             title='Taux de Maladie par Décile dAge et Genre',
             color_discrete_map={'M': 'steelblue', 'F': 'salmon'})
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

Pour les deux genres, le taux de maladie augmente avec l'âge. L'écart genre × maladie persiste dans la plupart des tranches d'âge, ce qui confirme qu'il ne s'agit pas d'un simple paradoxe de Simpson. Le biais de genre est structurel dans ce dataset.

### Analyse bivariée sur View position

In [ ]:
view_dist = df_clean['View Position'].value_counts()
taux_pa = df_clean[df_clean['View Position'] == 'PA']['Has_Disease'].mean()
taux_ap = df_clean[df_clean['View Position'] == 'AP']['Has_Disease'].mean() if 'AP' in df_clean['View Position'].values else 0
print(f"  Position PA : {taux_pa:.4f}")
print(f"  Position AP : {taux_ap:.4f}")

  Position PA : 0.3243
  Position AP : 0.4067


La position PA affiche un taux de maladie de 32,43%. Ce chiffre est très proche de la moyenne globale de 33,86% que nous avons calculée précédemment, ce qui est logique puisque ce groupe constitue la majorité de l'échantillon. La position AP, présente un taux de maladie significativement plus élevé de 40,67%. Ce qui est coherent car les patients en position AP sont généralement plus graves et donc plus susceptibles de présenter des maladies.

#3. Metrique de fairness

### 1. Metrique de fairness pour le genre

Le code de get_metrics est issus des td

In [ ]:
y_true_s  = pd.Series(df_clean['Has_Disease'].values,   name='Has_Disease')
prot_s    = pd.Series(df_clean['Gender_Binary'].values,  name='Gender_Binary')
metrics_orig = get_metrics(
    y_true=y_true_s,
    y_pred=y_true_s,
    prot_attr=prot_s,
    priv_group=1,
    pos_label=1,
    sample_weight=None
)

In [ ]:
base_rate_genre = metrics_orig['base_rate_truth'] * 100
spd_genre = metrics_orig['statistical_parity_difference']
di_genre = metrics_orig['disparate_impact_ratio']

print(f"\n Taux de base global : {base_rate_genre:.2f}%")
print(f"Statistical Parity Difference : {spd_genre:.4f}")
print(f"Disparate Impact Ratio : {di_genre:.4f}")



 Taux de base global : 33.86%
Statistical Parity Difference : -0.0013
Disparate Impact Ratio : 0.9962


Dans l'ensemble de la population, environ 33,86% des personnes sont malades.
Nous constatons que la différence de taux de maladie entre le groupe privilégié et non-privilégié est très proche de zéro avec -0,13%. Le SPD est inferieur a 0,05.

Le ratio des taux de maladie entre les groupes est de 0,9962 (entre 0,8 et 1,25). Cela veut dire que le groupe non-privilégié a 99,62% du taux du groupe privilégié.

Les données ne présentent pas de biais significatif.  Nous n'avons n'avons presque pas de biais dans nos données.


### Faut-il tout de même faire du pré-processing pour Réduire les Biais qui comme nous l'avons dit precedemment est quasi nul?

Oui car si nous souhaitons entraîner un modèle, même un petit biais peut être amplifié par le modèle de machine learning lors de l'apprentissage.
Le but de notre pré-traitement peut donc être préventif.

## 2. Metrique de fairness pour l'age

precedemment nous avions utiliser les deciles pour traiter les diffrenetes categorie d'age par dizaines, toutefois, notre fonction get_metric est une focntion binaire, je choisis alors de faire une comparaison entre deux groupe extreme, les jeunes et les âgés.
nous allons utiliser la mediane comme seuil.

In [ ]:
age_median = df_clean['Patient Age'].median()
df_clean['old_age'] = (df_clean['Patient Age'] > age_median).astype(int)

print(f"age médian : {age_median:.1f} ans")
print(f"Groupe 1 (en-dessous de la médiane) : {(df_clean['old_age'] == 0).sum()} patients")
print(f"Groupe 2 (au-dessus de la médiane) : {(df_clean['old_age'] == 1).sum()} patients")

y_age = pd.Series(df_clean['Has_Disease'].values, name='Has_Disease')
prot_age = pd.Series(df_clean['old_age'].values, name='old_age')

metrics_age = get_metrics(
    y_true=y_age,
    y_pred=y_age,
    prot_attr=prot_age,
    priv_group=1,  # le groupe au-dessus de la médiane est vu comme privilégié
    pos_label=1,
    sample_weight=None
)


age médian : 48.0 ans
Groupe 1 (en-dessous de la médiane) : 7700 patients
Groupe 2 (au-dessus de la médiane) : 7298 patients


In [ ]:
base_rate_age = metrics_age['base_rate_truth'] * 100
spd_age = metrics_age['statistical_parity_difference']
di_age = metrics_age['disparate_impact_ratio']

print(f"Taux de base global : {base_rate_age:.2f}%")
print(f"Statistical Parity Difference : {spd_age:.4f}")
print(f"Disparate Impact Ratio : {di_age:.4f}")


Taux de base global : 33.86%
Statistical Parity Difference : -0.1400
Disparate Impact Ratio : 0.6590


La spd de -0,14 montre un déséquilibre important entre les deux groupes d'âge comparés. en effet, cela signifie que les personnes de plus de 48 ans dans notre configuration, présente un taux de maladie inférieur à celui du groupe non privilégié des moins de 48 ans Ce chiffre dépasse le seuil considéré comme acceptable (1,25), ce qui indique un biais significatif dans les données.
Le Disparate Impact Ratio de 0,6590 confirme et amplifie ce constat. Cette valeur signifie que le groupe des moins de 48 ans n'a que 65,9% du taux de maladie du groupe des plus de 48 ans, ce qui est bien en dessous du seuil minimal de 0,8. En d'autres termes, les personnes âgées de plus de 48 ans ont un risque de maladie environ 52% plus élevé que les plus jeunes, un écart considérable qui ne peut être ignoré.

## 3. Metrique de fairness pour View Position

In [ ]:
metrics_view = get_metrics(
    y_true=df_clean['Has_Disease'].values,
    y_pred=df_clean['Has_Disease'].values,
    prot_attr=df_clean['View_Binary'].values,
    priv_group=1,
    pos_label=1,
    sample_weight=None
)
print(f"Statistical Parity Difference : {metrics_view['statistical_parity_difference']:.4f}")
print(f"Disparate Impact Ratio : {metrics_view['disparate_impact_ratio']:.4f}")

Statistical Parity Difference : -0.0824
Disparate Impact Ratio : 0.7973


Le spd de -0,0824 indique que le groupe AP présente un taux de maladie inférieur de 8,24 points de pourcentage par rapport au groupe PA, une différence qui se situe juste en dessous du seuil de 0,1 généralement considéré comme acceptable. Le dir de 0,7973 confirme cette tendance en montrant que le groupe AP n'a que 79,7% du taux de maladie du groupe PA, une valeur très légèrement en dessous du seuil

#  Méthodes de Pré-processing pour Réduire les Biais - méthode de mitigation des biais par pré-processing

### 4. Méthode : Reweighing (Re-pondération)

Methode du Reweighting ou au lieu de changer le nombre de personnes, on change leur "importance" c'est a dire leur poids dans mon dataset !
Comme ça, le modèle va "voir" autant d'hommes malades que de femmes malades, sans supprimer personne !

### Code de reweighting expliquer par rapoort au td2 et au cours 4

In [ ]:
weights = np.ones(len(df_clean))
n_total = len(df_clean)

In [ ]:
def compute_reweighing_weights(df, sensitive_attr, label_col):
    weights = np.ones(len(df))
    n_total = len(df)
#on parcourt les valeurs possible de l'attribut sensible et otutes les valeurs possibles du label
    for s_val in df[sensitive_attr].unique():
        for y_val in df[label_col].unique():
        #on veut uniquement conserver les lignes qui correspondent à cette combinaison
            select = (df[sensitive_attr] == s_val) & (df[label_col] == y_val)
            p_sy = select.sum() / n_total
            p_s  = (df[sensitive_attr] == s_val).sum() / n_total  #
            p_y  = (df[label_col] == y_val).sum() / n_total
            if p_sy > 0:
              #reweighting sous-représenté, poids>1 et sur-représenté, le poids < 1
                w_val = (p_s * p_y) / p_sy
                # Assigner le poids aux lignes correspondantes
                loc_idx = np.where(select.values)[0]
                weights[loc_idx] = w_val
    return weights


### Et sur nos données ?

### Reweighting par rapport au genre

In [ ]:
df_clean['sample_weight'] = compute_reweighing_weights(df_clean, 'Gender_Binary', 'Has_Disease')

Distribution des poids

In [ ]:
print(df_clean['sample_weight'].describe())

count    14998.000000
mean         1.000000
std          0.001344
min          0.998280
25%          0.998953
50%          1.000883
75%          1.000883
max          1.002051
Name: sample_weight, dtype: float64


### Comparaison

In [ ]:
metrics_avant = get_metrics(
    y_true=df_clean['Has_Disease'].values,
    y_pred=df_clean['Has_Disease'].values,
    prot_attr=df_clean['Gender_Binary'].values,
    priv_group=1,
    pos_label=1,
    sample_weight=None
)
metrics_apres = get_metrics(
    y_true=df_clean['Has_Disease'].values,
    y_pred=df_clean['Has_Disease'].values,
    prot_attr=df_clean['Gender_Binary'].values,
    priv_group=1,
    pos_label=1,
    sample_weight=df_clean['sample_weight'].values
)

In [ ]:

print("Statistical Parity Difference :")
print(f"  Avant Reweighing : {metrics_avant['statistical_parity_difference']:.4f}")
print(f"  Après Reweighing  : {metrics_apres['statistical_parity_difference']:.4f}")

print("\nDisparate Impact Ratio :")
print(f"  Avant Reweighing : {metrics_avant['disparate_impact_ratio']:.4f}")
print(f"  Après Reweighing  : {metrics_apres['disparate_impact_ratio']:.4f}")




Statistical Parity Difference :
  Avant Reweighing : -0.0013
  Après Reweighing  : -0.0000

Disparate Impact Ratio :
  Avant Reweighing : 0.9962
  Après Reweighing  : 1.0000


Le Reweighing a parfaitement atteint son objectif, les taux de maladie sont maintenant rigoureusement égaux entre hommes et femmes.

### Reweighting par rapport au genre


In [ ]:
df_clean['age_weights'] = compute_reweighing_weights(
    df_clean,
    'old_age',
    'Has_Disease'
)


Distribution des poids

In [ ]:
print(df_clean['age_weights'].describe())

count    14998.000000
mean         1.000000
std          0.149882
min          0.824911
25%          0.906611
50%          0.906611
75%          1.121936
max          1.251833
Name: age_weights, dtype: float64


In [ ]:
metrics_age_apres = get_metrics(
    y_true=y_age,
    y_pred=y_age,
    prot_attr=prot_age,
    priv_group=1,
    pos_label=1,
    sample_weight=df_clean['age_weights'].values
)


print("Statistical Parity Difference :")
print(f"  Avant Reweighing : {metrics_age_apres['statistical_parity_difference']:.4f}")

print("\nDisparate Impact Ratio :")
print(f"  Avant Reweighing : {metrics_age_apres['disparate_impact_ratio']:.4f}")

Statistical Parity Difference :
  Avant Reweighing : -0.0000

Disparate Impact Ratio :
  Avant Reweighing : 1.0000


### Reweighting par rapport au View Position

In [ ]:
df_clean['view_weights'] = compute_reweighing_weights(
    df_clean,
    'View_Binary',
    'Has_Disease'
)
y_view = pd.Series(df_clean['Has_Disease'].values, name='Has_Disease')
prot_view = pd.Series(df_clean['View_Binary'].values, name='View_Binary')

metrics_view_apres = get_metrics(
    y_true=y_view,
    y_pred=y_view,
    prot_attr=prot_view,
    priv_group=1,
    pos_label=1,
    sample_weight=df_clean['view_weights'].values
)
print(f"   Statistical Parity Difference : {metrics_view_apres['statistical_parity_difference']:.4f}")
print(f"   Disparate Impact Ratio : {metrics_view_apres['disparate_impact_ratio']:.4f}")


   Statistical Parity Difference : 0.0000
   Disparate Impact Ratio : 1.0000


Le Reweighing a parfaitement corrigé le biais lié à la position de vue.

# Conclusion

Dans le cadre de ce mi-projet, dont l’objectif était d’analyser les métadonnées du dataset NIH Chest X-Ray et de mettre en place un pré-processing permettant de réduire les biais, le travail réalisé a permis d’identifier puis de corriger efficacement un biais de genre présent dans les taux de maladie.

Après nettoyage des données, traitement des valeurs aberrantes et suppression des doublons patients Ce dernier point était crucial car un même patient apparaissant plusieurs fois aurait faussé l'ensemble des analyses statistiques et des métriques de fairness.

L'étude des biais potentiels s'est concentrée sur trois attributs sensibles majeurs. Pour le genre, les métriques ont révélé une situation quasi idéale avec un Statistical Parity Difference de -0,0013 et un Disparate Impact Ratio de 0,9962. L'âge a présenté un biais plus marqué avec un SPD de -0,1400 et un DI de 0,6590, signifiant que les patients de plus de 48 ans avaient un taux de maladie supérieur de 14 points. La position de vue a montré un biais modéré avec un SPD de -0,0824 et un DI de 0,7973, accompagné d'un déséquilibre numérique puisque seulement 17,4% des patients étaient en position AP contre 82,6% en position PA.

La méthode du Reweighing a été appliquée avec succès aux trois attributs sensibles.

Pour la suite, il conviendra d'entraîner un modèle de classification sur les images en utilisant les poids calculés pour garantir des prédictions équitables